# Arkadia — Montecarlo para sábado 7 de noviembre de 2026
**Datos sintéticos, no observación real.** Se parte del Excel del viernes 6. Se usa Poisson por intervalos de cinco minutos, con variaciones de demanda y semilla reproducible. Los topes no representan capacidad física medida.
Sin tiempos de servicio no se puede calcular un modelo M/M/1 o M/M/s completo.


In [ ]:
import zipfile, xml.etree.ElementTree as ET, random, math, statistics, datetime, csv
from google.colab import files
print("Sube datos del 6 nov.xlsx")
uploaded=files.upload(); ruta=next(iter(uploaded))
ns={'x':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
with zipfile.ZipFile(ruta) as z:
    root=ET.fromstring(z.read('xl/worksheets/sheet1.xml'))
    ss=ET.fromstring(z.read('xl/sharedStrings.xml'))
    strings=[''.join(t.text or '' for t in si.findall('.//x:t',ns)) for si in ss.findall('x:si',ns)]
    registros=[]
    for row in root.findall('.//x:sheetData/x:row',ns):
        if int(row.attrib['r'])==1: continue
        celdas={}
        for c in row.findall('x:c',ns):
            v=c.find('x:v',ns)
            if v is not None:
                col=''.join(ch for ch in c.attrib['r'] if ch.isalpha())
                celdas[col]=strings[int(v.text)] if c.attrib.get('t')=='s' else float(v.text)
        registros.append([celdas[k] for k in 'CDEF'])
assert len(registros)==24, 'Se esperaban 24 intervalos de cinco minutos'
SEMILLA=20261107
AUMENTOS=[1.25,1.15,1.30,1.20] # hipótesis: E70, S70, E80, S80
rng=random.Random(SEMILLA)
def poisson(lam):
    if lam<=0:return 0
    p=1.; k=0; limite=math.exp(-lam)
    while p>limite:
        k+=1;p*=rng.random()
    return k-1
# 18:00-19:30 proviene de viernes; 19:30-20:00 se extrapola
base=registros[6:]+[[statistics.mean(r[j] for r in registros[18:24]) for j in range(4)] for _ in range(6)]
topes=[max(1,int(max(r[j] for r in registros)*1.35)) for j in range(4)]
salida=[]
for i,b in enumerate(base):
    instante=datetime.datetime(2026,11,7,18,0)+datetime.timedelta(minutes=5*i)
    valores=[]
    for j,media in enumerate(b):
        lam=media*AUMENTOS[j]*(1+0.10*math.sin(i*math.pi/12))
        if rng.random()<0.045: lam*=0.18
        valores.append(min(poisson(lam),topes[j]))
    salida.append([instante.strftime('%H:%M'),(instante+datetime.timedelta(minutes=5)).strftime('%H:%M')]+valores+[valores[0]+valores[2],valores[1]+valores[3]])
encabezados=['Desde','Hasta','Entrada 70','Salida 70','Entrada 80','Salida 80','Entradas total','Salidas total']
with open('arkadia_sabado_7_simulado.csv','w',newline='',encoding='utf-8-sig') as f:
    writer=csv.writer(f);writer.writerow(encabezados);writer.writerows(salida)
print('SIMULACIÓN SINTÉTICA, NO MEDICIÓN REAL')
print('Topes supuestos:',topes)
print('Totales simulados (E70,S70,E80,S80):',[sum(r[2+j] for r in salida) for j in range(4)])
files.download('arkadia_sabado_7_simulado.csv')
